# Agentic Financial Research Assistant

### Project Overview

The Agentic Financial Research Assistant is a project I developed to make working with financial documents easier. Instead of manually searching through lengthy PDF reports, users can simply ask questions and get answers based on the information in the document.

Along with answering questions, the assistant can also perform mathematical calculations and convert currencies. It uses a RAG-based approach to find relevant information from financial documents and a LangChain agent that decides which tool to use depending on the user's request.

### Project Objectives

- Make it easier to find information from financial PDF documents.
- Use embeddings and ChromaDB to retrieve relevant information based on user queries.
- Build an AI agent that can choose and use different tools automatically.
- Add a calculator for mathematical operations.
- Integrate a currency converter to handle currency-related queries.
- Use conversation memory so the assistant can remember previous interactions.
- Maintain separate conversations using unique thread IDs.

### Technologies Used

Python, LangChain, Gemini, Hugging Face Embeddings, ChromaDB, LangGraph and REST API.

In [1]:
!pip install -qU langchain_classic langchain_google_genai langchain_community langchain_huggingface langchain_chroma
!pip install -qU tiktoken pypdf chromadb sentence-transformers
!pip install -q langchain langgraph

In [2]:
import os
import ast
import operator
import requests
import warnings

warnings.filterwarnings("ignore")

from google.colab import userdata, drive

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import MultiQueryRetriever

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

GEMINI_KEY = userdata.get("PROJECT_KEY")

if not GEMINI_KEY:
    raise ValueError("PROJECT_KEY not found in Colab Secrets.")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=GEMINI_KEY,
    temperature=0
)

print("Gemini LLM initialized successfully!")

Gemini LLM initialized successfully!


In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
pdf_path = "/content/drive/MyDrive/gen_ai_rag_assignment/Financial_Research_RAG_Assistant.pdf"

pdf_loader = PyPDFLoader(pdf_path)

pdf_docs = pdf_loader.load()

print("PDF loaded successfully!")
print("Total pages:", len(pdf_docs))

PDF loaded successfully!
Total pages: 18


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    length_function=len
)

chunks = text_splitter.split_documents(pdf_docs)

print("Document splitting completed!")
print("Total chunks:", len(chunks))

Document splitting completed!
Total chunks: 43


In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="finance_research_agent"
)

print("ChromaDB vector database created successfully!")
print("Total vectors:", vectorstore._collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB vector database created successfully!
Total vectors: 43


In [7]:
# ============================================================
# TOOL 1: RAG TOOL
# ============================================================

@tool
def rag_tool(query: str) -> str:
    """
    Answer questions using information from the financial
    research PDF knowledge base.
    """

    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}
    )

    multi_query = MultiQueryRetriever.from_llm(
        retriever=retriever,
        llm=llm
    )

    docs = multi_query.invoke(query)

    if not docs:
        return "Sorry, I can't answer this question."

    # Remove duplicate chunks
    unique_docs = []
    seen = set()

    for doc in docs:
        key = doc.page_content.strip()

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    # Limit retrieved context
    unique_docs = unique_docs[:6]

    # Prepare context with page numbers
    context_parts = []

    for doc in unique_docs:
        page = doc.metadata.get("page")

        if page is not None:
            page_number = page + 1
            page_label = f"Page {page_number}"
        else:
            page_label = "Page number unavailable"

        context_parts.append(
            f"[{page_label}]\n{doc.page_content}"
        )

    context = "\n\n".join(context_parts)

    # RAG prompt
    template = """
    You are a financial research assistant.

    Answer the question using ONLY the provided context.

    Rules:
    1. Do not use outside knowledge.
    2. Do not invent facts, figures, or statistics.
    3. Cite relevant pages using [Page N].
    4. If the answer is unavailable in the context,
       say: "Sorry, I can't answer this question."
    5. Use clear bullet points where appropriate.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    prompt = PromptTemplate(
        input_variables=["context", "query"],
        template=template
    )

    rag_chain = prompt | llm | StrOutputParser()

    answer = rag_chain.invoke({
        "context": context,
        "query": query
    })

    return answer


# ============================================================
# TOOL 2: CALCULATOR
# ============================================================

# Safe arithmetic operations (without using eval)
operators = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod
}


def safe_calculate(node):

    if isinstance(node, ast.Expression):
        return safe_calculate(node.body)

    elif isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numbers are allowed.")

    elif isinstance(node, ast.BinOp):
        op_type = type(node.op)

        if op_type not in operators:
            raise ValueError("Unsupported mathematical operation.")

        return operators[op_type](
            safe_calculate(node.left),
            safe_calculate(node.right)
        )

    elif isinstance(node, ast.UnaryOp):
        value = safe_calculate(node.operand)

        if isinstance(node.op, ast.UAdd):
            return value

        elif isinstance(node.op, ast.USub):
            return -value

        raise ValueError("Unsupported unary operation.")

    elif isinstance(node, ast.Call):
        if (
            isinstance(node.func, ast.Name)
            and node.func.id == "round"
            and 1 <= len(node.args) <= 2
            and not node.keywords
        ):
            args = [safe_calculate(arg) for arg in node.args]
            return round(*args)

        raise ValueError("Only round() is supported.")

    else:
        raise ValueError("Invalid mathematical expression.")


@tool
def calculator(expression: str) -> str:
    """Calculate mathematical expressions using arithmetic operations."""

    try:
        tree = ast.parse(expression, mode="eval")
        result = safe_calculate(tree)

        return str(result)

    except Exception as e:
        return f"Calculation error: {str(e)}"


# ============================================================
# TOOL 3: CURRENCY CONVERTER
# ============================================================

@tool
def currency_converter(
    amount: float,
    from_currency: str,
    to_currency: str
) -> str:
    """
    Convert an amount between currencies using the latest
    available exchange rates.
    """

    try:
        from_currency = from_currency.upper().strip()
        to_currency = to_currency.upper().strip()

        url = (
            f"https://open.er-api.com/v6/latest/{from_currency}"
        )

        response = requests.get(url, timeout=15)

        response.raise_for_status()

        data = response.json()

        if data.get("result") != "success":
            return "Unable to retrieve exchange rates."

        rates = data.get("rates", {})

        if to_currency not in rates:
            return f"Currency {to_currency} is not supported."

        rate = rates[to_currency]
        converted_amount = amount * rate

        return (
            f"{amount:.2f} {from_currency} = "
            f"{converted_amount:.2f} {to_currency}\n"
            f"Exchange rate: 1 {from_currency} = "
            f"{rate} {to_currency}"
        )

    except Exception as e:
        return f"Currency conversion failed: {str(e)}"


# Store exactly three tools
tools = [
    rag_tool,
    calculator,
    currency_converter
]

print("Three tools created successfully!")

for tool_item in tools:
    print("-", tool_item.name)

Three tools created successfully!
- rag_tool
- calculator
- currency_converter


In [8]:
checkpointer = InMemorySaver()

print("Memory checkpointer created successfully!")

Memory checkpointer created successfully!


In [9]:
agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=checkpointer,
    system_prompt="""
    You are a helpful AI assistant with access to three tools.

    TOOL SELECTION RULES:

    1. RAG TOOL:
       Use rag_tool only when the user asks a question
       about the financial research PDF.

    2. CALCULATOR:
       Use calculator whenever mathematical calculations
       are required.

    3. CURRENCY CONVERTER:
       Use currency_converter whenever currency conversion
       is required.

    4. GENERAL CONVERSATION:
       For greetings, introductions, personal information,
       memory-related questions, and general conversation,
       respond directly without calling any tool.

    Automatically select the appropriate tool when needed.
    Do not call unrelated tools.

    Provide clear and concise answers.
    """
)

print("Agent created successfully!")

Agent created successfully!


In [10]:
config = {
    "configurable": {
        "thread_id": "user_1"
    }
}

print("Conversation thread user_1 created.")

Conversation thread user_1 created.


In [11]:
def run_agent(query, config):

    # Get the number of messages already in this thread
    previous_state = agent.get_state(config)

    previous_messages = (
        previous_state.values.get("messages", [])
        if previous_state and previous_state.values
        else []
    )

    previous_count = len(previous_messages)

    # Invoke agent
    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        },
        config=config
    )

    # Only consider messages from the current interaction
    new_messages = response["messages"][previous_count:]

    print("User:", query)

    print("\nTools Used:")

    tool_calls_found = False

    for message in new_messages:

        if getattr(message, "tool_calls", None):

            for call in message.tool_calls:
                print("-", call["name"])
                tool_calls_found = True

    if not tool_calls_found:
        print("No tool call detected.")

    print("\nAgent Response:")

    final_content = response["messages"][-1].content

    if isinstance(final_content, list):

        for block in final_content:

            if isinstance(block, dict) and block.get("type") == "text":
                print(block["text"])

    else:
        print(final_content)

    print("\n" + "=" * 60)

    return None

In [12]:
run_agent(
    "What is (15.5 * 24) / 3.2 rounded to 2 decimals?",
    config
)

User: What is (15.5 * 24) / 3.2 rounded to 2 decimals?

Tools Used:
- calculator

Agent Response:
(15.5 * 24) / 3.2 = **116.25**



In [13]:
run_agent(
    "Convert 100 USD to INR.",
    config
)

User: Convert 100 USD to INR.

Tools Used:
- currency_converter

Agent Response:
100 USD is equivalent to **9,508.25 INR** (at an exchange rate of 1 USD = 95.0825 INR).



In [14]:
run_agent(
    "What is the purpose of an Annual Report?",
    config
)

User: What is the purpose of an Annual Report?

Tools Used:
- rag_tool

Agent Response:
Based on the financial report knowledge base, the purpose of an Annual Report is:

* **Comprehensive Overview:** It serves as a comprehensive corporate document describing a company's performance, financial condition, business activities, risks, and management perspective for a specific reporting period.
* **Combined Insights:** It combines narrative discussions with audited financial statements and explanatory notes, providing stakeholders with a detailed and transparent understanding of the company.



In [15]:
run_agent(
    "My name is Ronit. I am learning Generative AI.",
    config
)

User: My name is Ronit. I am learning Generative AI.

Tools Used:
No tool call detected.

Agent Response:
Nice to meet you, Ronit! It's great that you're learning Generative AI—it's a rapidly evolving and exciting field. Whether you're exploring large language models, prompt engineering, computer vision, or building AI-powered applications, feel free to ask if you have any questions or need help with concepts along the way!



In [16]:
run_agent(
    "What is my name and what am I learning?",
    config
)

User: What is my name and what am I learning?

Tools Used:
No tool call detected.

Agent Response:
Your name is **Ronit**, and you are learning **Generative AI**!



In [17]:
run_agent(
    "What were the previous questions I asked you?",
    config
)

User: What were the previous questions I asked you?

Tools Used:
No tool call detected.

Agent Response:
Here are the previous questions and prompts you asked:

1. **"What is (15.5 * 24) / 3.2 rounded to 2 decimals?"**
2. **"Convert 100 USD to INR."**
3. **"What is the purpose of an Annual Report?"**
4. **"My name is Ronit. I am learning Generative AI."** *(Introduction)*
5. **"What is my name and what am I learning?"**



In [18]:
config_user2 = {
    "configurable": {
        "thread_id": "user_2"
    }
}

print("New conversation thread user_2 created.")

New conversation thread user_2 created.


In [19]:
run_agent(
    "What is my name and what am I learning?",
    config_user2
)

User: What is my name and what am I learning?

Tools Used:
No tool call detected.

Agent Response:
I don't have access to your personal information or past interactions, so I don't know your name or what you are currently learning. Feel free to share them with me!

